# Name That LLM — UltraBERT v2 checkpoint 2 submission

Offline inference using the private Kaggle Model variation `zodrak/ultrabert380-v2/pytorch/checkpoint2-pytorch/1`. This is the current v2 training run’s selected best checkpoint at epoch 2 (validation accuracy 0.533333). The existing private dataset is attached only to provide the shared offline inference code.

In [ ]:
from pathlib import Path
import json, os, shutil, sys, zipfile

MODEL_INPUT = Path('/kaggle/input/models/zodrak/ultrabert380-v2/pytorch/checkpoint2-pytorch/1')
MODEL_DIR = MODEL_INPUT / 'model'
HELPER_DATASET = Path('/kaggle/input/ultrabert380-v15-best-epoch5')
HELPER_PACKAGE = HELPER_DATASET / 'model_package.zip'
HELPER_ROOT = Path('/kaggle/working/nlm_ultrabert_inference_helpers')
TEST_CANDIDATES = [Path('/kaggle/input/competitions/name-that-llm/test.csv'), Path('/kaggle/input/name-that-llm/test.csv')]
OUTPUT = Path('/kaggle/working/submission.parquet')
EXPECTED_EXPERIMENT_ID = 'ultrabert380_v2_data88cea686_split59c9631c_seed2026_cfg5635d548_code667e2d0'
EXPECTED_EPOCH = 2
EXPECTED_VALIDATION_ACCURACY = 0.5333333333333333

assert MODEL_DIR.is_dir(), f'Attached Kaggle model files not found: {MODEL_DIR}'
assert (MODEL_DIR / 'model.safetensors').is_file(), 'UltraBERT weights are missing'
assert (MODEL_DIR / 'tokenizer.json').is_file(), 'UltraBERT tokenizer is missing'
assert HELPER_PACKAGE.is_file(), f'Inference helper package not found: {HELPER_PACKAGE}'
TEST_CSV = next((path for path in TEST_CANDIDATES if path.is_file()), None)
assert TEST_CSV is not None, f'Competition test.csv not found in: {TEST_CANDIDATES}'

if HELPER_ROOT.exists():
    shutil.rmtree(HELPER_ROOT)
HELPER_ROOT.mkdir(parents=True)
with zipfile.ZipFile(HELPER_PACKAGE) as package:
    package.extractall(HELPER_ROOT)
sys.path.insert(0, str(HELPER_ROOT))

model_config = json.loads((MODEL_DIR / 'config.json').read_text())
classes = [model_config['id2label'][str(i)] for i in range(len(model_config['id2label']))]
selection = json.loads((MODEL_DIR / 'selection.json').read_text())
assert selection['epoch'] == EXPECTED_EPOCH, selection
assert abs(selection['value'] - EXPECTED_VALIDATION_ACCURACY) < 1e-12, selection
manifest = {
    'experiment_id': EXPECTED_EXPERIMENT_ID,
    'model': 'ultrabert380',
    'checkpoint': str(MODEL_DIR),
    'classes': classes,
    'selected_epoch': selection['epoch'],
    'selected_validation_accuracy': selection['value'],
    'training_dataset': 'v2',
    'config': {'max_length': 384},
    'calibration_temperature': 1.0,
}
MANIFEST_PATH = Path('/kaggle/working/ultrabert380_v2_checkpoint2_manifest.json')
MANIFEST_PATH.write_text(json.dumps(manifest, indent=2) + '\n')

os.environ['HF_HUB_OFFLINE'] = '1'
os.environ['TRANSFORMERS_OFFLINE'] = '1'
import torch
print({'torch': torch.__version__, 'cuda_build': torch.version.cuda, 'cuda_available': torch.cuda.is_available()})
assert torch.cuda.is_available(), 'Kaggle did not provide a CUDA GPU; aborting before inference.'
print({'experiment_id': manifest['experiment_id'], 'selected_epoch': manifest['selected_epoch'],
       'validation_accuracy': manifest['selected_validation_accuracy'], 'model_path': str(MODEL_DIR),
       'device': 'cuda', 'test': str(TEST_CSV)})

In [ ]:
import pandas as pd
from src.inference import export_submission

result = export_submission('ultrabert380', MANIFEST_PATH, TEST_CSV, OUTPUT)
submission = pd.read_parquet(OUTPUT)
test = pd.read_csv(TEST_CSV, dtype={'id': 'string'})
assert submission.columns.tolist() == ['id', 'model']
assert len(submission) == len(test)
assert submission['id'].astype(str).tolist() == test['id'].astype(str).tolist()
assert submission['id'].notna().all() and not submission['id'].duplicated().any()
assert submission['model'].notna().all()
assert submission['model'].isin(manifest['classes']).all()
print({'rows': len(submission), 'output': str(OUTPUT), 'model': result['model'],
       'experiment_id': manifest['experiment_id'], 'selected_epoch': manifest['selected_epoch'],
       'selected_validation_accuracy': manifest['selected_validation_accuracy']})
display(submission.head())